#### Check if GPU is available (Mac)

In [57]:
import torch
import torch.nn as nn
from torch.nn import functional as F

if torch.backends.mps.is_available():
    mps_device = torch.device("mps")
    x = torch.ones(1, device=mps_device)
    print (x)
    # output expected:
    # tensor([1.], device='mps:0')
else:
    print ("MPS device not found.")

# Initialize hyperparams
block_size = 8
batch_size = 4

tensor([1.], device='mps:0')


In [58]:
with open('wizard_of_oz.txt', 'r', encoding='utf-8') as f:
    text = f.read()

chars = sorted(set(text))
print(chars)

vocab_size = len(chars)
print(vocab_size)

['\n', ' ', '!', '"', '&', "'", '(', ')', '*', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '\ufeff']
81


In [39]:
string_to_int = { ch:i for i, ch in enumerate(chars)}
int_to_string = { i:ch for i, ch in enumerate(chars)}
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])

# print(encode('\n abdullah'))
# print(decode([61, 58, 65, 65, 68]))
# Output ->
# [0, 1, 54, 55, 57, 74, 65, 65, 54, 61]
# hello

data = torch.tensor(encode(text), dtype=torch.long)
print(data[:100])

# print(decode([80, 28, 39, 42, 39, 44, 32, 49,  1, 25, 38, 28,  1, 44, 32, 29,  1, 47,
#         33, 50, 25, 42, 28,  1, 33, 38,  1, 39, 50,  0,  0,  1,  1, 26, 49,  0,
#          0,  1,  1, 36, 11,  1, 30, 42, 25, 38, 35,  1, 26, 25, 45, 37,  0,  0,
#          1,  1, 25, 45, 44, 32, 39, 42,  1, 39, 30,  1, 44, 32, 29,  1, 47, 33,
#         50, 25, 42, 28,  1, 39, 30,  1, 39, 50,  9,  1, 44, 32, 29,  1, 36, 25,
#         38, 28,  1, 39, 30,  1, 39, 50,  9,  1]))


tensor([80, 28, 39, 42, 39, 44, 32, 49,  1, 25, 38, 28,  1, 44, 32, 29,  1, 47,
        33, 50, 25, 42, 28,  1, 33, 38,  1, 39, 50,  0,  0,  1,  1, 26, 49,  0,
         0,  1,  1, 36, 11,  1, 30, 42, 25, 38, 35,  1, 26, 25, 45, 37,  0,  0,
         1,  1, 25, 45, 44, 32, 39, 42,  1, 39, 30,  1, 44, 32, 29,  1, 47, 33,
        50, 25, 42, 28,  1, 39, 30,  1, 39, 50,  9,  1, 44, 32, 29,  1, 36, 25,
        38, 28,  1, 39, 30,  1, 39, 50,  9,  1])


#### Choose 80% of tensor length for training

In [56]:
n = int(0.8*len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    print(ix)
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(mps_device), y.to(mps_device)
    return x,y

x, y = get_batch('train')
print('inputs:')
print(x)
print('targets:')
print(y)


tensor([119612,  28095,  95635,  96225])
inputs:
tensor([[ 1, 76, 54, 65, 65, 72, 11,  0],
        [67,  1, 73, 61, 58,  1, 76, 54],
        [58, 71, 72,  1, 61, 54, 57,  1],
        [ 1, 74, 69, 68, 67,  1, 73, 61]], device='mps:0')
targets:
tensor([[76, 54, 65, 65, 72, 11,  0,  0],
        [ 1, 73, 61, 58,  1, 76, 54, 78],
        [71, 72,  1, 61, 54, 57,  1, 55],
        [74, 69, 68, 67,  1, 73, 61, 58]], device='mps:0')


In [42]:

x = train_data[:block_size]
y = train_data[1:block_size+1]
print(f'First {block_size} = {x}')
print(f'Next {block_size} = {y}')

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f'when input is {context} target is {target}')

First 8 = tensor([80, 28, 39, 42, 39, 44, 32, 49])
Next 8 = tensor([28, 39, 42, 39, 44, 32, 49,  1])
when input is tensor([80]) target is 28
when input is tensor([80, 28]) target is 39
when input is tensor([80, 28, 39]) target is 42
when input is tensor([80, 28, 39, 42]) target is 39
when input is tensor([80, 28, 39, 42, 39]) target is 44
when input is tensor([80, 28, 39, 42, 39, 44]) target is 32
when input is tensor([80, 28, 39, 42, 39, 44, 32]) target is 49
when input is tensor([80, 28, 39, 42, 39, 44, 32, 49]) target is 1


In [53]:
torch.randint(10, (2,3)).float()

tensor([[1., 7., 0.],
        [8., 5., 9.]])

In [ ]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size) -> None:
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, index, targets):
        logits = self.token_embedding_table(index)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss
            